In [7]:
import sys
sys.path.insert(0, r"e:\TalentLens")

In [2]:
import os
import torch
import pandas as pd
from datasets import load_dataset
from transformers import BertTokenizer

In [6]:
from src.data.preprocess import clean_text, tokenize_texts
from src.utils.helpers import load_label_maps

In [2]:
dataset = load_dataset(
    "cnamuangtoun/resume-job-description-fit",
    cache_dir="./hf_cache"
)

train_df = dataset['train'].to_pandas()
test_df  = dataset['test'].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

Train shape: (6241, 3)
Test shape:  (1759, 3)


In [3]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove non-ASCII characters (PDF artifacts, weird bullets, fancy quotes)
    text = text.encode('ascii', errors='ignore').decode('ascii')
    # Normalize all whitespace (newlines, tabs, multiple spaces) to single space
    text = re.sub(r'\s+', ' ', text)
    # Strip leading and trailing whitespace
    text = text.strip()
    # Lowercase
    text = text.lower()
    return text

# Apply to both columns in train and test
train_df['resume_clean'] = train_df['resume_text'].apply(clean_text)
train_df['jd_clean']     = train_df['job_description_text'].apply(clean_text)

test_df['resume_clean']  = test_df['resume_text'].apply(clean_text)
test_df['jd_clean']      = test_df['job_description_text'].apply(clean_text)

# Spot check — compare raw vs cleaned for 3 rows
for i in range(3):
    print(f"\n--- Row {i} ---")
    print(f"[RAW]\n{train_df['resume_text'].iloc[i][:300]}")
    print(f"[CLEAN]\n{train_df['resume_clean'].iloc[i][:300]}")


--- Row 0 ---
[RAW]
SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.
Highlights-Soft Skills: Public Speaking, Public Relati
[CLEAN]
summaryhighly motivated sales associate with extensive customer service and sales experience. outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market. highlights-soft skills: public speaking, public relati

--- Row 1 ---
[RAW]
Professional SummaryCurrently working with Caterpillar as an contract employee Active in several NPI and CPI projects Provide technical for CAT facilities worldwide 24-hours a day Willing to travel to meet job requirements Familiar with Power Distribution Systems, NETA, NFPA 70E, and OHSA electrical
[CLEAN]
professional summarycurrently working w

In [7]:
# Define the mapping — check these match your actual label strings from EDA
label2id = {
    "No Fit"       : 0,
    "Potential Fit" : 1,
    "Good Fit"      : 2
}
id2label = {v: k for k, v in label2id.items()}

print("label2id:", label2id)
print("id2label:", id2label)

# Apply to train and test
train_df['label_id'] = train_df['label'].map(label2id)
test_df['label_id']  = test_df['label'].map(label2id)

# Sanity check — should have no NaN if all label strings matched
print("\nTrain null label_ids:", train_df['label_id'].isnull().sum())
print("Test  null label_ids:", test_df['label_id'].isnull().sum())

# Confirm distribution is preserved
print("\nTrain label_id distribution:")
print(train_df['label_id'].value_counts().sort_index())

label2id: {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}
id2label: {0: 'No Fit', 1: 'Potential Fit', 2: 'Good Fit'}

Train null label_ids: 0
Test  null label_ids: 0

Train label_id distribution:
label_id
0    3143
1    1556
2    1542
Name: count, dtype: int64


In [8]:
# Load the tokenizer
# Using bert-base-uncased — same weights we'll load in Week 3 for the model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LENGTH = 512  # each of resume and JD gets the full 512 independently (bi-encoder advantage)

def tokenize_texts(df, tokenizer, max_length):
    """
    Tokenize resume and JD separately.
    Returns a dict of tensors ready for the bi-encoder.
    """
    resume_encodings = tokenizer(
        df['resume_clean'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    jd_encodings = tokenizer(
        df['jd_clean'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    labels = torch.tensor(df['label_id'].tolist(), dtype=torch.long)

    return {
        'resume_input_ids'      : resume_encodings['input_ids'],
        'resume_attention_mask' : resume_encodings['attention_mask'],
        'jd_input_ids'          : jd_encodings['input_ids'],
        'jd_attention_mask'     : jd_encodings['attention_mask'],
        'labels'                : labels
    }

print("Tokenizing train...")
train_encodings = tokenize_texts(train_df, tokenizer, MAX_LENGTH)

print("Tokenizing test...")
test_encodings  = tokenize_texts(test_df, tokenizer, MAX_LENGTH)

# Confirm shapes — every tensor should match (num_rows, 512)
print("\n=== Train tensor shapes ===")
for k, v in train_encodings.items():
    print(f"  {k}: {v.shape}")

print("\n=== Test tensor shapes ===")
for k, v in test_encodings.items():
    print(f"  {k}: {v.shape}")

Tokenizing train...
Tokenizing test...

=== Train tensor shapes ===
  resume_input_ids: torch.Size([6241, 512])
  resume_attention_mask: torch.Size([6241, 512])
  jd_input_ids: torch.Size([6241, 512])
  jd_attention_mask: torch.Size([6241, 512])
  labels: torch.Size([6241])

=== Test tensor shapes ===
  resume_input_ids: torch.Size([1759, 512])
  resume_attention_mask: torch.Size([1759, 512])
  jd_input_ids: torch.Size([1759, 512])
  jd_attention_mask: torch.Size([1759, 512])
  labels: torch.Size([1759])


In [9]:
def truncation_rate(input_ids, tokenizer):
    """
    A sequence was truncated if its last token is NOT a padding token.
    PAD token id for bert-base-uncased is 0.
    """
    PAD_ID = tokenizer.pad_token_id
    last_tokens = input_ids[:, -1]  # last token of every sequence
    truncated   = (last_tokens != PAD_ID).sum().item()
    total       = input_ids.shape[0]
    return truncated, total, round(truncated / total * 100, 2)

r_trunc, r_total, r_pct = truncation_rate(train_encodings['resume_input_ids'], tokenizer)
j_trunc, j_total, j_pct = truncation_rate(train_encodings['jd_input_ids'], tokenizer)

print(f"Resumes truncated : {r_trunc}/{r_total} ({r_pct}%)")
print(f"JDs truncated     : {j_trunc}/{j_total} ({j_pct}%)")

Resumes truncated : 5836/6241 (93.51%)
JDs truncated     : 2781/6241 (44.56%)


In [10]:
os.makedirs("../data/processed", exist_ok=True)

torch.save(train_encodings, "../data/processed/train_encodings.pt")
torch.save(test_encodings,  "../data/processed/test_encodings.pt")

# Save label mappings too — you'll need these in Week 5 for the UI
import json
with open("../data/processed/label2id.json", "w") as f:
    json.dump(label2id, f)
with open("../data/processed/id2label.json", "w") as f:
    json.dump(id2label, f)

print("Saved:")
for fname in os.listdir("../data/processed"):
    size = os.path.getsize(f"../data/processed/{fname}")
    print(f"  {fname} — {size / 1024 / 1024:.1f} MB")

Saved:
  .gitkeep — 0.0 MB
  id2label.json — 0.0 MB
  label2id.json — 0.0 MB
  test_encodings.pt — 27.5 MB
  train_encodings.pt — 97.6 MB
